In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/silver-helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F
df_sprints = (spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id))

In [0]:
df_sprints_dropped = df_sprints.drop('url')

In [0]:
df_sprints_final = (
    df_sprints_dropped.withColumnsRenamed({
        'constructorId' : 'constructor_id',
        'driverId' : 'driver_id',
        'raceName' : 'race_name',
        'positionText' : 'finish_position_text',
        'date' : 'race_date',
        'grid' : 'grid_position',
        'laps' : 'completed_laps',
        'number' : 'car_number',
        'position' :'finish_position'
    })
    .filter(F.col('season').isNotNull() &
            F.col('round').isNotNull() &
            F.col('constructor_id').isNotNull() &
            F.col('driver_id').isNotNull())
    .dropDuplicates(['season' , 'round' , 'constructor_id' , 'driver_id'])
    .withColumn('race_name', F.initcap(F.col('race_name')))
)

In [0]:
write_to_silver(input_df=df_sprints_final,
                target_table=silver_table,
                merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
                columns_to_update=[
                    "race_name",
                    "race_date",
                    "grid_position",
                    "completed_laps",
                    "car_number",
                    "points",
                    "finish_position",
                    "finish_position_text",
                    "status",
                    "ingestion_timestamp",
                    "source_file",
                    "batch_id"
                    ]
)